In [1]:
from pathlib import Path
import random
import copy
import gc

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import (
    TensorDataset,
    DataLoader,
)

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    precision_recall_curve,
)

In [2]:
SEEDS = [
    42,
    52,
    62,
    72,
    82,
]

BATCH_SIZE = 512

SOURCE_DIM = 203
TARGET_DIM = 69
LATENT_DIM = 64

# Source pretraining
SOURCE_PRETRAIN_LR = 1e-3

# HDA
LR_SOURCE = 1e-4
LR_TARGET = 1e-3
LR_CLASSIFIER = 1e-4

WEIGHT_DECAY = 1e-4

LAMBDA_MMD = 0.01

MAX_EPOCHS = 30
PATIENCE = 5

MMD_EVAL_BATCHES = 20

In [3]:
ROOT = next(
    (
        p
        for p in [
            Path.cwd(),
            *Path.cwd().parents,
        ]
        if (
            (p / "data").is_dir()
            and
            (p / "notebooks").is_dir()
        )
    ),
    Path.cwd(),
)

HDA_DIR = (
    ROOT
    / "data"
    / "processed"
    / "hda"
)

MODEL_DIR = (
    ROOT
    / "models"
    / "multiseed"
)

RESULT_DIR = (
    ROOT
    / "results"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")

elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")

else:
    DEVICE = torch.device("cpu")


print("Device:", DEVICE)
print("HDA_DIR:", HDA_DIR)
print("MODEL_DIR:", MODEL_DIR)

Device: mps
HDA_DIR: /Users/thonph/Desktop/KLTN/data/processed/hda
MODEL_DIR: /Users/thonph/Desktop/KLTN/models/multiseed


In [4]:
def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

In [5]:
# ============================================================
# SOURCE
# ============================================================

X_s_train = np.load(
    HDA_DIR / "X_s_train.npy",
    mmap_mode="c",
)

X_s_val = np.load(
    HDA_DIR / "X_s_val.npy",
    mmap_mode="c",
)

X_s_test = np.load(
    HDA_DIR / "X_s_test.npy",
    mmap_mode="c",
)

y_s_train = np.load(
    HDA_DIR / "y_s_train.npy"
)

y_s_val = np.load(
    HDA_DIR / "y_s_val.npy"
)

y_s_test = np.load(
    HDA_DIR / "y_s_test.npy"
)


# ============================================================
# TARGET
# ============================================================

X_t_train = np.load(
    HDA_DIR / "X_t_train.npy",
    mmap_mode="c",
)

X_t_test = np.load(
    HDA_DIR / "X_t_test.npy",
    mmap_mode="c",
)

# Evaluation only
y_t_test = np.load(
    HDA_DIR / "y_t_test.npy"
)


print("Source train:", X_s_train.shape)
print("Source val:", X_s_val.shape)
print("Source test:", X_s_test.shape)

print("Target adaptation:", X_t_train.shape)
print("Target test:", X_t_test.shape)

Source train: (1441582, 203)
Source val: (308911, 203)
Source test: (308911, 203)
Target adaptation: (2259960, 69)
Target test: (564991, 69)


In [6]:
assert X_s_train.shape[1] == SOURCE_DIM
assert X_s_val.shape[1] == SOURCE_DIM
assert X_s_test.shape[1] == SOURCE_DIM

assert X_t_train.shape[1] == TARGET_DIM
assert X_t_test.shape[1] == TARGET_DIM

assert X_s_train.dtype == np.float32
assert X_t_train.dtype == np.float32

assert np.isfinite(X_s_train).all()
assert np.isfinite(X_t_train).all()

print("Fixed data split validated.")

Fixed data split validated.


In [7]:
class SourceEncoder(nn.Module):

    def __init__(
        self,
        input_dim,
        latent_dim=64,
    ):
        super().__init__()

        self.encoder = nn.Sequential(

            nn.Linear(
                input_dim,
                256,
            ),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(
                256,
                128,
            ),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(
                128,
                latent_dim,
            ),
        )

    def forward(self, x):
        return self.encoder(x)

In [8]:
class TargetEncoder(nn.Module):

    def __init__(
        self,
        input_dim,
        latent_dim=64,
    ):
        super().__init__()

        self.encoder = nn.Sequential(

            nn.Linear(
                input_dim,
                256,
            ),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(
                256,
                128,
            ),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(
                128,
                latent_dim,
            ),
        )

    def forward(self, x):
        return self.encoder(x)

In [9]:
class BinaryClassifier(nn.Module):

    def __init__(
        self,
        latent_dim=64,
    ):
        super().__init__()

        self.classifier = nn.Sequential(

            nn.Linear(
                latent_dim,
                32,
            ),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(
                32,
                1,
            ),
        )

    def forward(self, z):

        return (
            self.classifier(z)
            .squeeze(1)
        )

In [10]:
def pairwise_sq_dist(
    x,
    y,
):

    x_norm = (
        x.pow(2)
        .sum(
            dim=1,
            keepdim=True,
        )
    )

    y_norm = (
        y.pow(2)
        .sum(
            dim=1,
            keepdim=True,
        )
        .T
    )

    dist = (
        x_norm
        + y_norm
        - 2.0 * x @ y.T
    )

    return dist.clamp_min(0.0)

In [11]:
@torch.no_grad()
def median_bandwidth(
    source,
    target,
):

    combined = torch.cat(
        [source, target],
        dim=0,
    )

    distances = pairwise_sq_dist(
        combined,
        combined,
    )

    n = distances.size(0)

    mask = ~torch.eye(
        n,
        dtype=torch.bool,
        device=distances.device,
    )

    values = distances[mask]

    sigma2 = torch.median(
        values
    )

    return sigma2.clamp_min(
        1e-6
    )

In [12]:
def rbf_kernel(
    x,
    y,
    sigma2,
):

    distances = pairwise_sq_dist(
        x,
        y,
    )

    return torch.exp(
        -distances
        /
        (2.0 * sigma2)
    )

In [13]:
def mmd_rbf(
    source,
    target,
):

    n_s = source.size(0)
    n_t = target.size(0)

    sigma2 = median_bandwidth(
        source.detach(),
        target.detach(),
    )

    K_ss = rbf_kernel(
        source,
        source,
        sigma2,
    )

    K_tt = rbf_kernel(
        target,
        target,
        sigma2,
    )

    K_st = rbf_kernel(
        source,
        target,
        sigma2,
    )

    source_term = (
        K_ss.sum()
        - torch.diagonal(K_ss).sum()
    ) / (
        n_s * (n_s - 1)
    )

    target_term = (
        K_tt.sum()
        - torch.diagonal(K_tt).sum()
    ) / (
        n_t * (n_t - 1)
    )

    cross_term = K_st.mean()

    return (
        source_term
        + target_term
        - 2.0 * cross_term
    )

In [14]:
def build_datasets():

    source_train_dataset = TensorDataset(
        torch.from_numpy(
            np.asarray(X_s_train)
        ),
        torch.from_numpy(
            y_s_train.astype(
                np.float32
            )
        ),
    )

    source_val_dataset = TensorDataset(
        torch.from_numpy(
            np.asarray(X_s_val)
        ),
        torch.from_numpy(
            y_s_val.astype(
                np.float32
            )
        ),
    )

    source_test_dataset = TensorDataset(
        torch.from_numpy(
            np.asarray(X_s_test)
        ),
        torch.from_numpy(
            y_s_test.astype(
                np.float32
            )
        ),
    )

    target_train_dataset = TensorDataset(
        torch.from_numpy(
            np.asarray(X_t_train)
        ),
    )

    target_test_dataset = TensorDataset(
        torch.from_numpy(
            np.asarray(X_t_test)
        ),
        torch.from_numpy(
            y_t_test.astype(
                np.float32
            )
        ),
    )

    return (
        source_train_dataset,
        source_val_dataset,
        source_test_dataset,
        target_train_dataset,
        target_test_dataset,
    )

In [15]:
def build_loaders(seed):

    (
        source_train_dataset,
        source_val_dataset,
        source_test_dataset,
        target_train_dataset,
        target_test_dataset,
    ) = build_datasets()


    source_generator = (
        torch.Generator()
        .manual_seed(seed)
    )

    target_generator = (
        torch.Generator()
        .manual_seed(seed + 10_000)
    )


    source_train_loader = DataLoader(
        source_train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=True,
        num_workers=0,
        generator=source_generator,
    )


    target_train_loader = DataLoader(
        target_train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=True,
        num_workers=0,
        generator=target_generator,
    )


    source_val_loader = DataLoader(
        source_val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=0,
    )


    source_test_loader = DataLoader(
        source_test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=0,
    )


    target_test_loader = DataLoader(
        target_test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=0,
    )


    # Fixed loaders only for MMD comparison
    source_mmd_loader = DataLoader(
        source_train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=True,
        num_workers=0,
    )


    target_mmd_loader = DataLoader(
        target_train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=True,
        num_workers=0,
    )


    return {
        "source_train":
            source_train_loader,

        "target_train":
            target_train_loader,

        "source_val":
            source_val_loader,

        "source_test":
            source_test_loader,

        "target_test":
            target_test_loader,

        "source_mmd":
            source_mmd_loader,

        "target_mmd":
            target_mmd_loader,
    }

In [16]:
@torch.no_grad()
def predict_domain(
    encoder,
    classifier,
    loader,
):

    encoder.eval()
    classifier.eval()

    all_probs = []
    all_labels = []

    for x, y in loader:

        x = x.to(
            DEVICE,
            dtype=torch.float32,
        )

        z = encoder(x)

        logits = classifier(z)

        probs = torch.sigmoid(
            logits
        )

        all_probs.append(
            probs.cpu().numpy()
        )

        all_labels.append(
            y.numpy()
        )

    return (
        np.concatenate(all_labels),
        np.concatenate(all_probs),
    )

In [17]:
def classification_metrics(
    y_true,
    y_prob,
    threshold=0.5,
):

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    return {
        "pr_auc":
            average_precision_score(
                y_true,
                y_prob,
            ),

        "roc_auc":
            roc_auc_score(
                y_true,
                y_prob,
            ),

        "precision":
            precision_score(
                y_true,
                y_pred,
                zero_division=0,
            ),

        "recall":
            recall_score(
                y_true,
                y_pred,
                zero_division=0,
            ),

        "f1":
            f1_score(
                y_true,
                y_pred,
                zero_division=0,
            ),

        "confusion_matrix":
            confusion_matrix(
                y_true,
                y_pred,
            ),
    }

In [18]:
def select_threshold(
    y_true,
    y_prob,
):

    precision_vals, recall_vals, thresholds = (
        precision_recall_curve(
            y_true,
            y_prob,
        )
    )

    f1_vals = (
        2
        * precision_vals[:-1]
        * recall_vals[:-1]
        /
        (
            precision_vals[:-1]
            + recall_vals[:-1]
            + 1e-12
        )
    )

    best_idx = np.argmax(
        f1_vals
    )

    return (
        float(
            thresholds[best_idx]
        ),
        float(
            f1_vals[best_idx]
        ),
    )

In [19]:
def source_train_epoch(
    encoder,
    classifier,
    loader,
    criterion,
    optimizer,
):

    encoder.train()
    classifier.train()

    total_loss = 0.0
    total_samples = 0


    for x, y in loader:

        x = x.to(
            DEVICE,
            dtype=torch.float32,
        )

        y = y.to(
            DEVICE,
            dtype=torch.float32,
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        z = encoder(x)

        logits = classifier(z)

        loss = criterion(
            logits,
            y,
        )

        loss.backward()

        optimizer.step()

        n = x.size(0)

        total_loss += (
            loss.item() * n
        )

        total_samples += n


    return (
        total_loss
        / total_samples
    )

In [20]:
def pretrain_source(
    seed,
    loaders,
    pos_weight,
):

    set_seed(seed)


    source_encoder = SourceEncoder(
        SOURCE_DIM,
        LATENT_DIM,
    ).to(DEVICE)

    classifier = BinaryClassifier(
        LATENT_DIM
    ).to(DEVICE)


    criterion = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight
    )


    optimizer = torch.optim.AdamW(
        list(
            source_encoder.parameters()
        )
        +
        list(
            classifier.parameters()
        ),
        lr=SOURCE_PRETRAIN_LR,
        weight_decay=WEIGHT_DECAY,
    )


    best_val_pr_auc = -np.inf

    best_state = None

    epochs_without_improvement = 0


    for epoch in range(
        1,
        MAX_EPOCHS + 1,
    ):

        train_loss = source_train_epoch(
            source_encoder,
            classifier,
            loaders["source_train"],
            criterion,
            optimizer,
        )


        y_val_true, y_val_prob = (
            predict_domain(
                source_encoder,
                classifier,
                loaders["source_val"],
            )
        )


        val_pr_auc = (
            average_precision_score(
                y_val_true,
                y_val_prob,
            )
        )


        if (
            val_pr_auc
            > best_val_pr_auc
        ):

            best_val_pr_auc = (
                val_pr_auc
            )

            epochs_without_improvement = 0

            best_state = {
                "epoch":
                    epoch,

                "source_encoder":
                    copy.deepcopy(
                        source_encoder.state_dict()
                    ),

                "classifier":
                    copy.deepcopy(
                        classifier.state_dict()
                    ),
            }

        else:

            epochs_without_improvement += 1


        if (
            epochs_without_improvement
            >= PATIENCE
        ):
            break


    assert best_state is not None


    source_encoder.load_state_dict(
        best_state[
            "source_encoder"
        ]
    )

    classifier.load_state_dict(
        best_state[
            "classifier"
        ]
    )


    # Source test baseline
    y_test_true, y_test_prob = (
        predict_domain(
            source_encoder,
            classifier,
            loaders["source_test"],
        )
    )


    source_metrics = (
        classification_metrics(
            y_test_true,
            y_test_prob,
        )
    )


    return {
        "encoder":
            source_encoder,

        "classifier":
            classifier,

        "best_epoch":
            best_state["epoch"],

        "best_val_pr_auc":
            best_val_pr_auc,

        "source_test_metrics":
            source_metrics,
    }

In [21]:
@torch.no_grad()
def estimate_latent_mmd(
    source_encoder,
    target_encoder,
    source_loader,
    target_loader,
    max_batches=20,
):

    source_encoder.eval()
    target_encoder.eval()

    values = []

    source_iter = iter(
        source_loader
    )

    target_iter = iter(
        target_loader
    )


    n_batches = min(
        len(source_loader),
        len(target_loader),
        max_batches,
    )


    for _ in range(n_batches):

        xs, _ = next(
            source_iter
        )

        (xt,) = next(
            target_iter
        )


        xs = xs.to(
            DEVICE,
            dtype=torch.float32,
        )

        xt = xt.to(
            DEVICE,
            dtype=torch.float32,
        )


        zs = source_encoder(
            xs
        )

        zt = target_encoder(
            xt
        )


        values.append(
            mmd_rbf(
                zs,
                zt,
            ).item()
        )


    return float(
        np.mean(values)
    )

In [22]:
def train_hda_epoch(
    source_encoder,
    target_encoder,
    classifier,
    source_loader,
    target_loader,
    criterion,
    optimizer,
):

    source_encoder.train()
    target_encoder.train()
    classifier.train()

    total_loss = 0.0
    total_cls = 0.0
    total_mmd = 0.0


    n_steps = max(
        len(source_loader),
        len(target_loader),
    )


    source_iter = iter(
        source_loader
    )

    target_iter = iter(
        target_loader
    )


    for _ in range(n_steps):

        try:

            xs, ys = next(
                source_iter
            )

        except StopIteration:

            source_iter = iter(
                source_loader
            )

            xs, ys = next(
                source_iter
            )


        try:

            (xt,) = next(
                target_iter
            )

        except StopIteration:

            target_iter = iter(
                target_loader
            )

            (xt,) = next(
                target_iter
            )


        xs = xs.to(
            DEVICE,
            dtype=torch.float32,
        )

        ys = ys.to(
            DEVICE,
            dtype=torch.float32,
        )

        xt = xt.to(
            DEVICE,
            dtype=torch.float32,
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        zs = source_encoder(xs)

        zt = target_encoder(xt)


        logits_s = classifier(
            zs
        )


        loss_cls = criterion(
            logits_s,
            ys,
        )


        loss_mmd = mmd_rbf(
            zs,
            zt,
        )


        loss = (
            loss_cls
            +
            LAMBDA_MMD
            * loss_mmd
        )


        loss.backward()

        optimizer.step()


        total_loss += loss.item()

        total_cls += (
            loss_cls.item()
        )

        total_mmd += (
            loss_mmd.item()
        )


    return {
        "loss":
            total_loss / n_steps,

        "classification":
            total_cls / n_steps,

        "mmd":
            total_mmd / n_steps,
    }

In [23]:
def run_hda_seed(
    seed,
    loaders,
    pretrained,
    pos_weight,
):

    set_seed(seed)


    source_encoder = pretrained[
        "encoder"
    ]


    classifier = pretrained[
        "classifier"
    ]


    # New random TargetEncoder
    target_encoder = TargetEncoder(
        TARGET_DIM,
        LATENT_DIM,
    ).to(DEVICE)


    criterion = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight
    )


    optimizer = torch.optim.AdamW(
        [
            {
                "params":
                    source_encoder.parameters(),

                "lr":
                    LR_SOURCE,
            },

            {
                "params":
                    target_encoder.parameters(),

                "lr":
                    LR_TARGET,
            },

            {
                "params":
                    classifier.parameters(),

                "lr":
                    LR_CLASSIFIER,
            },
        ],
        weight_decay=WEIGHT_DECAY,
    )


    # --------------------------------------------------------
    # Initial MMD
    # --------------------------------------------------------

    initial_mmd = estimate_latent_mmd(
        source_encoder,
        target_encoder,
        loaders["source_mmd"],
        loaders["target_mmd"],
        MMD_EVAL_BATCHES,
    )


    # --------------------------------------------------------
    # HDA training
    # --------------------------------------------------------

    best_val_pr_auc = -np.inf

    best_state = None

    epochs_without_improvement = 0


    for epoch in range(
        1,
        MAX_EPOCHS + 1,
    ):

        train_metrics = train_hda_epoch(
            source_encoder,
            target_encoder,
            classifier,
            loaders["source_train"],
            loaders["target_train"],
            criterion,
            optimizer,
        )


        y_val_true, y_val_prob = (
            predict_domain(
                source_encoder,
                classifier,
                loaders["source_val"],
            )
        )


        val_pr_auc = (
            average_precision_score(
                y_val_true,
                y_val_prob,
            )
        )


        if (
            val_pr_auc
            > best_val_pr_auc
        ):

            best_val_pr_auc = (
                val_pr_auc
            )

            epochs_without_improvement = 0


            best_state = {
                "epoch":
                    epoch,

                "source_encoder":
                    copy.deepcopy(
                        source_encoder.state_dict()
                    ),

                "target_encoder":
                    copy.deepcopy(
                        target_encoder.state_dict()
                    ),

                "classifier":
                    copy.deepcopy(
                        classifier.state_dict()
                    ),
            }

        else:

            epochs_without_improvement += 1


        if (
            epochs_without_improvement
            >= PATIENCE
        ):

            break


    assert best_state is not None


    source_encoder.load_state_dict(
        best_state[
            "source_encoder"
        ]
    )

    target_encoder.load_state_dict(
        best_state[
            "target_encoder"
        ]
    )

    classifier.load_state_dict(
        best_state[
            "classifier"
        ]
    )


    # --------------------------------------------------------
    # Threshold = SOURCE VALIDATION ONLY
    # --------------------------------------------------------

    y_val_true, y_val_prob = (
        predict_domain(
            source_encoder,
            classifier,
            loaders["source_val"],
        )
    )


    decision_threshold, val_f1 = (
        select_threshold(
            y_val_true,
            y_val_prob,
        )
    )


    # --------------------------------------------------------
    # Source test
    # --------------------------------------------------------

    y_source_true, y_source_prob = (
        predict_domain(
            source_encoder,
            classifier,
            loaders["source_test"],
        )
    )


    source_metrics = (
        classification_metrics(
            y_source_true,
            y_source_prob,
            decision_threshold,
        )
    )


    # --------------------------------------------------------
    # Target test
    # TARGET LABELS FIRST USED HERE
    # --------------------------------------------------------

    y_target_true, y_target_prob = (
        predict_domain(
            target_encoder,
            classifier,
            loaders["target_test"],
        )
    )


    target_metrics = (
        classification_metrics(
            y_target_true,
            y_target_prob,
            decision_threshold,
        )
    )


    target_prevalence = float(
        y_target_true.mean()
    )


    # --------------------------------------------------------
    # Final MMD
    # --------------------------------------------------------

    final_mmd = estimate_latent_mmd(
        source_encoder,
        target_encoder,
        loaders["source_mmd"],
        loaders["target_mmd"],
        MMD_EVAL_BATCHES,
    )


    if abs(initial_mmd) > 1e-12:

        mmd_reduction_pct = (
            (
                initial_mmd
                - final_mmd
            )
            /
            abs(initial_mmd)
            * 100.0
        )

    else:

        mmd_reduction_pct = np.nan


    return {
        "source_encoder":
            source_encoder,

        "target_encoder":
            target_encoder,

        "classifier":
            classifier,

        "best_epoch":
            best_state["epoch"],

        "best_val_pr_auc":
            best_val_pr_auc,

        "decision_threshold":
            decision_threshold,

        "source_metrics":
            source_metrics,

        "target_metrics":
            target_metrics,

        "target_prevalence":
            target_prevalence,

        "initial_mmd":
            initial_mmd,

        "final_mmd":
            final_mmd,

        "mmd_reduction_pct":
            mmd_reduction_pct,
    }

In [24]:
n_negative = int(
    (y_s_train == 0).sum()
)

n_positive = int(
    (y_s_train == 1).sum()
)

POS_WEIGHT_VALUE = (
    n_negative
    / n_positive
)

pos_weight = torch.tensor(
    [POS_WEIGHT_VALUE],
    dtype=torch.float32,
    device=DEVICE,
)


print("Normal:", n_negative)
print("Attack:", n_positive)
print("pos_weight:", POS_WEIGHT_VALUE)

Normal: 1371832
Attack: 69750
pos_weight: 19.66784229390681


In [25]:
all_results = []


for seed in SEEDS:

    print("\n")
    print("=" * 70)
    print(
        f"RUNNING SEED {seed}"
    )
    print("=" * 70)


    # --------------------------------------------------------
    # Seed + loaders
    # --------------------------------------------------------

    set_seed(seed)

    loaders = build_loaders(
        seed
    )


    # --------------------------------------------------------
    # 1. Source pretraining
    # --------------------------------------------------------

    print(
        "[1/2] Source pretraining..."
    )

    pretrained = pretrain_source(
        seed,
        loaders,
        pos_weight,
    )


    pretrained_metrics = (
        pretrained[
            "source_test_metrics"
        ]
    )


    print(
        "Pretrained source PR-AUC:",
        pretrained_metrics[
            "pr_auc"
        ]
    )


    # --------------------------------------------------------
    # 2. HDA + MMD
    # --------------------------------------------------------

    print(
        "[2/2] HDA + MMD..."
    )

    hda = run_hda_seed(
        seed,
        loaders,
        pretrained,
        pos_weight,
    )


    source_metrics = (
        hda["source_metrics"]
    )

    target_metrics = (
        hda["target_metrics"]
    )


    source_pr_auc_change = (
        source_metrics["pr_auc"]
        -
        pretrained_metrics["pr_auc"]
    )


    target_above_prevalence = (
        target_metrics["pr_auc"]
        >
        hda["target_prevalence"]
    )


    result = {

        "seed":
            seed,

        # --------------------
        # Source pretraining
        # --------------------

        "pretrain_best_epoch":
            pretrained[
                "best_epoch"
            ],

        "pretrained_source_pr_auc":
            pretrained_metrics[
                "pr_auc"
            ],

        "pretrained_source_roc_auc":
            pretrained_metrics[
                "roc_auc"
            ],

        # --------------------
        # HDA
        # --------------------

        "hda_best_epoch":
            hda[
                "best_epoch"
            ],

        "source_pr_auc":
            source_metrics[
                "pr_auc"
            ],

        "source_roc_auc":
            source_metrics[
                "roc_auc"
            ],

        "source_f1":
            source_metrics[
                "f1"
            ],

        "source_pr_auc_change":
            source_pr_auc_change,

        # --------------------
        # Target
        # --------------------

        "target_pr_auc":
            target_metrics[
                "pr_auc"
            ],

        "target_roc_auc":
            target_metrics[
                "roc_auc"
            ],

        "target_precision":
            target_metrics[
                "precision"
            ],

        "target_recall":
            target_metrics[
                "recall"
            ],

        "target_f1":
            target_metrics[
                "f1"
            ],

        "target_prevalence":
            hda[
                "target_prevalence"
            ],

        "target_above_prevalence":
            target_above_prevalence,

        # --------------------
        # MMD
        # --------------------

        "initial_mmd":
            hda[
                "initial_mmd"
            ],

        "final_mmd":
            hda[
                "final_mmd"
            ],

        "mmd_reduction_pct":
            hda[
                "mmd_reduction_pct"
            ],

        "decision_threshold":
            hda[
                "decision_threshold"
            ],
    }


    all_results.append(
        result
    )


    print(
        f"Seed {seed} completed"
    )

    print(
        "Target PR-AUC:",
        result[
            "target_pr_auc"
        ]
    )

    print(
        "Target ROC-AUC:",
        result[
            "target_roc_auc"
        ]
    )

    print(
        "MMD reduction:",
        result[
            "mmd_reduction_pct"
        ]
    )


    # --------------------------------------------------------
    # Save per-seed HDA model
    # --------------------------------------------------------

    checkpoint_path = (
        MODEL_DIR
        /
        f"hda_mmd_seed_{seed}.pt"
    )


    torch.save(
        {
            "seed":
                seed,

            "source_dim":
                SOURCE_DIM,

            "target_dim":
                TARGET_DIM,

            "latent_dim":
                LATENT_DIM,

            "lambda_mmd":
                LAMBDA_MMD,

            "pretrain_best_epoch":
                pretrained[
                    "best_epoch"
                ],

            "hda_best_epoch":
                hda[
                    "best_epoch"
                ],

            "decision_threshold":
                hda[
                    "decision_threshold"
                ],

            "source_encoder_state_dict":
                hda[
                    "source_encoder"
                ].state_dict(),

            "target_encoder_state_dict":
                hda[
                    "target_encoder"
                ].state_dict(),

            "classifier_state_dict":
                hda[
                    "classifier"
                ].state_dict(),

            "result":
                result,
        },

        checkpoint_path,
    )


    # --------------------------------------------------------
    # Save progress after every seed
    # --------------------------------------------------------

    pd.DataFrame(
        all_results
    ).to_csv(
        RESULT_DIR
        / "mmd_multiseed_results.csv",
        index=False,
    )


    # --------------------------------------------------------
    # Memory cleanup
    # --------------------------------------------------------

    del hda
    del pretrained
    del loaders

    gc.collect()

    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

    elif torch.cuda.is_available():
        torch.cuda.empty_cache()



RUNNING SEED 42
[1/2] Source pretraining...
Pretrained source PR-AUC: 0.9788121301242382
[2/2] HDA + MMD...
Seed 42 completed
Target PR-AUC: 0.1505167089595193
Target ROC-AUC: 0.38695813890365716
MMD reduction: 99.61046083848309


RUNNING SEED 52
[1/2] Source pretraining...
Pretrained source PR-AUC: 0.978169913112298
[2/2] HDA + MMD...
Seed 52 completed
Target PR-AUC: 0.1714854552362313
Target ROC-AUC: 0.44253397338210865
MMD reduction: 99.80393712019143


RUNNING SEED 62
[1/2] Source pretraining...
Pretrained source PR-AUC: 0.9784461848101431
[2/2] HDA + MMD...
Seed 62 completed
Target PR-AUC: 0.17636351486952803
Target ROC-AUC: 0.37934580063163076
MMD reduction: 98.69114071184644


RUNNING SEED 72
[1/2] Source pretraining...
Pretrained source PR-AUC: 0.9782223865455756
[2/2] HDA + MMD...
Seed 72 completed
Target PR-AUC: 0.15610018732101877
Target ROC-AUC: 0.38014954799631095
MMD reduction: 99.75284763250707


RUNNING SEED 82
[1/2] Source pretraining...
Pretrained source PR-AUC: 0.9

In [26]:
results_df = pd.DataFrame(
    all_results
)

results_df

,seed,pretrain_best_epoch,pretrained_source_pr_auc,pretrained_source_roc_auc,hda_best_epoch,source_pr_auc,source_roc_auc,source_f1,source_pr_auc_change,target_pr_auc,target_roc_auc,target_precision,target_recall,target_f1,target_prevalence,target_above_prevalence,initial_mmd,final_mmd,mmd_reduction_pct,decision_threshold
0,42,30,0.978812,0.998869,12,0.980091,0.998928,0.910590,0.001279,0.150517,0.386958,0.028111,0.005399,0.009059,0.196681,False,0.931499,0.003629,99.610461,0.949077
1,52,24,0.978170,0.998830,29,0.980216,0.998925,0.910566,0.002046,0.171485,0.442534,0.000574,0.000126,0.000207,0.196681,False,0.951715,0.001866,99.803937,0.950179
2,62,29,0.978446,0.998845,18,0.980273,0.998941,0.909862,0.001827,0.176364,0.379346,0.044202,0.005858,0.010346,0.196681,False,0.879789,0.011515,98.691141,0.946570
3,72,29,0.978222,0.998831,30,0.980281,0.998941,0.910149,0.002059,0.156100,0.380150,0.028333,0.005444,0.009134,0.196681,False,0.940746,0.002325,99.752848,0.941714
4,82,30,0.978235,0.998840,29,0.980238,0.998910,0.911488,0.002002,0.136261,0.318349,0.000688,0.000144,0.000238,0.196681,False,0.951729,0.000766,99.919547,0.949156


In [27]:
display_cols = [
    "seed",

    "pretrained_source_pr_auc",
    "source_pr_auc",
    "source_pr_auc_change",

    "target_pr_auc",
    "target_roc_auc",
    "target_prevalence",
    "target_above_prevalence",

    "initial_mmd",
    "final_mmd",
    "mmd_reduction_pct",

    "hda_best_epoch",
]

results_df[
    display_cols
]

,seed,pretrained_source_pr_auc,source_pr_auc,source_pr_auc_change,target_pr_auc,target_roc_auc,target_prevalence,target_above_prevalence,initial_mmd,final_mmd,mmd_reduction_pct,hda_best_epoch
0,42,0.978812,0.980091,0.001279,0.150517,0.386958,0.196681,False,0.931499,0.003629,99.610461,12
1,52,0.978170,0.980216,0.002046,0.171485,0.442534,0.196681,False,0.951715,0.001866,99.803937,29
2,62,0.978446,0.980273,0.001827,0.176364,0.379346,0.196681,False,0.879789,0.011515,98.691141,18
3,72,0.978222,0.980281,0.002059,0.156100,0.380150,0.196681,False,0.940746,0.002325,99.752848,30
4,82,0.978235,0.980238,0.002002,0.136261,0.318349,0.196681,False,0.951729,0.000766,99.919547,29


In [28]:
summary_metrics = [
    "pretrained_source_pr_auc",
    "source_pr_auc",
    "source_pr_auc_change",

    "target_pr_auc",
    "target_roc_auc",
    "target_f1",

    "initial_mmd",
    "final_mmd",
    "mmd_reduction_pct",
]


summary_df = (
    results_df[
        summary_metrics
    ]
    .agg(
        [
            "mean",
            "std",
        ]
    )
    .T
)


summary_df[
    "mean ± std"
] = (
    summary_df["mean"]
    .map(
        lambda x:
        f"{x:.6f}"
    )
    +
    " ± "
    +
    summary_df["std"]
    .map(
        lambda x:
        f"{x:.6f}"
    )
)


summary_df

,mean,std,mean ± std
pretrained_source_pr_auc,0.978377,0.000265,0.978377 ± 0.000265
source_pr_auc,0.980220,0.000077,0.980220 ± 0.000077
source_pr_auc_change,0.001843,0.000329,0.001843 ± 0.000329
target_pr_auc,0.158145,0.016211,0.158145 ± 0.016211
target_roc_auc,0.381467,0.044015,0.381467 ± 0.044015
target_f1,0.005797,0.005114,0.005797 ± 0.005114
initial_mmd,0.931095,0.029902,0.931095 ± 0.029902
final_mmd,0.004020,0.004314,0.004020 ± 0.004314
mmd_reduction_pct,99.555587,0.495816,99.555587 ± 0.495816


In [32]:
# ============================================================
# MULTI-SEED PERFORMANCE SUMMARY
# ============================================================

n_seeds = len(results_df)

# Target PR-AUC relative to prevalence
n_above_prevalence = int(
    (
        results_df["target_pr_auc"]
        >
        results_df["target_prevalence"]
    ).sum()
)

n_below_or_equal_prevalence = int(
    (
        results_df["target_pr_auc"]
        <=
        results_df["target_prevalence"]
    ).sum()
)

# Source performance change after adaptation
n_source_improved = int(
    (
        results_df["source_pr_auc_change"] > 0
    ).sum()
)

print("=" * 65)
print("MULTI-SEED SUMMARY")
print("=" * 65)

print(f"Seeds: {n_seeds}")

print(
    "Target PR-AUC > prevalence:",
    f"{n_above_prevalence}/{n_seeds}"
)

print(
    "Target PR-AUC <= prevalence:",
    f"{n_below_or_equal_prevalence}/{n_seeds}"
)

print(
    "Source PR-AUC improved after adaptation:",
    f"{n_source_improved}/{n_seeds}"
)

MULTI-SEED SUMMARY
Seeds: 5
Target PR-AUC > prevalence: 0/5
Target PR-AUC <= prevalence: 5/5
Source PR-AUC improved after adaptation: 5/5


In [33]:
print("=" * 65)
print("SINGLE-RBF HDA-MMD — 5 SEED RESULTS")
print("=" * 65)

print(
    "Source PR-AUC:",
    f"{results_df['source_pr_auc'].mean():.6f}",
    "±",
    f"{results_df['source_pr_auc'].std():.6f}",
)

print(
    "Target PR-AUC:",
    f"{results_df['target_pr_auc'].mean():.6f}",
    "±",
    f"{results_df['target_pr_auc'].std():.6f}",
)

print(
    "Target ROC-AUC:",
    f"{results_df['target_roc_auc'].mean():.6f}",
    "±",
    f"{results_df['target_roc_auc'].std():.6f}",
)

print(
    "Initial MMD²:",
    f"{results_df['initial_mmd'].mean():.6f}",
    "±",
    f"{results_df['initial_mmd'].std():.6f}",
)

print(
    "Final MMD²:",
    f"{results_df['final_mmd'].mean():.6f}",
    "±",
    f"{results_df['final_mmd'].std():.6f}",
)

print(
    "MMD reduction:",
    f"{results_df['mmd_reduction_pct'].mean():.2f}%",
    "±",
    f"{results_df['mmd_reduction_pct'].std():.2f}%",
)

SINGLE-RBF HDA-MMD — 5 SEED RESULTS
Source PR-AUC: 0.980220 ± 0.000077
Target PR-AUC: 0.158145 ± 0.016211
Target ROC-AUC: 0.381467 ± 0.044015
Initial MMD²: 0.931095 ± 0.029902
Final MMD²: 0.004020 ± 0.004314
MMD reduction: 99.56% ± 0.50%


In [34]:
assert len(
    results_df
) == len(SEEDS)

assert (
    results_df[
        "seed"
    ]
    .tolist()
    == SEEDS
)

assert np.isfinite(
    results_df[
        "target_pr_auc"
    ]
).all()

assert np.isfinite(
    results_df[
        "target_roc_auc"
    ]
).all()

assert np.isfinite(
    results_df[
        "initial_mmd"
    ]
).all()

assert np.isfinite(
    results_df[
        "final_mmd"
    ]
).all()


print("=" * 65)
print("5-SEED HDA-MMD EXPERIMENT CHECK PASSED")
print("=" * 65)

print(
    "Results saved to:"
)

print(
    RESULT_DIR
    / "mmd_multiseed_results.csv"
)

5-SEED HDA-MMD EXPERIMENT CHECK PASSED
Results saved to:
/Users/thonph/Desktop/KLTN/results/mmd_multiseed_results.csv
